# Trabajo Práctico Integrador: Análisis de Desempeño y Gestión de Estudiantes

**Materia:** Análisis de Datos Inicial
**Carrera:** Tecnicatura Universitaria en Programación (TUP)

**Integrantes — Grupo 7:**
1. Jeremías Bontorno Pontis
2. Nicolas Andres Hassan Padoan Vargas
3. Luciano Andres Mas Cannizzo
4. Axel Esteban Mejias
5. Leandro Nicolas Nuñez Agostinho
6. Valentino Vernier

---

## Hito 1: Elección y Planteo

### Dataset elegido: `Calificaciones.csv`

Conjunto de datos académicos de la TUP correspondiente al cuatrimestre marzo–junio 2026. El archivo está en **formato largo**: cada fila representa **una entrega de un alumno en una actividad puntual** (TP, Quiz o Parcial).

| Característica | Valor |
|---|---|
| Origen | Sistema de gestión académica de la TUP (cuatrimestre 2026-1) |
| Granularidad | 1 fila = 1 entrega de un alumno en una actividad |
| Filas | 5.610 |
| Alumnos únicos | 700 |
| Comisiones | 9 (A1–A3 mañana, B1–B2 tarde, C1–C4 noche) |
| Actividades por alumno | 4 TPs + 3 Quizzes + 1 Parcial = 8 |

**Columnas:** `ID_Alumno`, `Nombre_Apellido`, `Edad`, `Genero`, `Email`, `Comision`, `Turno`, `Fecha_Inscripcion`, `Actividad`, `Tipo_Actividad`, `Fecha_Limite`, `Fecha_Entrega`, `Nota`, `Estado_Entrega`.

### Objetivos del análisis

1. **Patrones de abandono del parcial.** ¿Qué combinación de constancia y rendimiento previo anticipa que un alumno no se presente al parcial?
2. **Diferencias entre comisiones y turnos.** ¿Hay comisiones con tasa de aprobación significativamente menor?
3. **Evolución temporal del rendimiento.** ¿Cómo evolucionan la nota promedio y la tasa de entrega a lo largo del cuatrimestre?

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

RUTA_CSV    = os.path.join("..", "data", "Calificaciones.csv")
RUTA_SALIDA = os.path.join("..", "data", "datos_limpios.csv")

---
## Hito 2: ETL y Calidad de Datos

### 2.1 Carga y auditoría inicial

In [2]:
def cargar_dataset(ruta: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(ruta)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"No se encontró '{ruta}'.") from e
    print(f"[OK] Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas.")
    return df

def auditar(df: pd.DataFrame) -> None:
    print("--- Tipos de datos ---")
    print(df.dtypes)
    print("\n--- Nulos por columna ---")
    print(df.isna().sum())
    print(f"\n--- Filas duplicadas exactas: {df.duplicated().sum()}")

df_raw = cargar_dataset(RUTA_CSV)
display(df_raw.head())
auditar(df_raw)

### 2.2 Limpieza

In [3]:
def normalizar_strings(df):
    out = df.copy()
    out["Nombre_Apellido"] = out["Nombre_Apellido"].str.strip().str.title()
    out["Comision"]        = out["Comision"].str.strip().str.upper()
    out["Turno"]           = out["Turno"].str.strip().str.title()
    out["Tipo_Actividad"]  = out["Tipo_Actividad"].str.strip().str.title()
    return out

def unificar_genero(df):
    mapeo = {"F":"F","FEMENINO":"F","Femenino":"F","femenino":"F",
             "M":"M","MASCULINO":"M","Masculino":"M","X":"X"}
    out = df.copy()
    out["Genero"] = out["Genero"].astype(str).str.strip().map(mapeo).fillna("Otro")
    return out

def unificar_estado_entrega(df):
    def _mapear(v):
        if pd.isna(v): return "No Entrego"
        s = str(v).strip().lower()
        if s == "" or "no" in s: return "No Entrego"
        if "tarde" in s:  return "Tarde"
        if "tiempo" in s: return "A Tiempo"
        return "Otro"
    out = df.copy()
    out["Estado_Entrega"] = out["Estado_Entrega"].map(_mapear)
    return out

def parsear_nota(df):
    out = df.copy()
    out["Nota"] = pd.to_numeric(out["Nota"].astype(str).str.replace(",",".",regex=False), errors="coerce")
    return out

def parsear_fechas(df):
    out = df.copy()
    for col in ["Fecha_Inscripcion","Fecha_Limite","Fecha_Entrega"]:
        out[col] = pd.to_datetime(out[col], errors="coerce", format="mixed")
    return out

def remover_duplicados(df):
    antes = len(df)
    out = df.drop_duplicates().reset_index(drop=True)
    print(f"[OK] Duplicados removidos: {antes - len(out)}")
    return out

### 2.3 Tratamiento de outliers

In [4]:
def tratar_outliers_nota(df):
    out = df.copy()
    mascara = (out["Nota"] < 0) | (out["Nota"] > 10)
    out.loc[mascara, "Nota"] = np.nan
    print(f"[OK] Notas fuera de rango marcadas como NaN: {int(mascara.sum())}")
    return out

def tratar_outliers_edad(df):
    out = df.copy()
    edades_validas = out.loc[(out["Edad"] >= 16) & (out["Edad"] <= 70), "Edad"]
    q1, q3 = edades_validas.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf = max(16, q1 - 1.5*iqr)
    lim_sup = min(70, q3 + 1.5*iqr)
    mediana = edades_validas.median()
    mascara = (out["Edad"] < lim_inf) | (out["Edad"] > lim_sup)
    out.loc[mascara, "Edad"] = mediana
    print(f"[OK] Edades fuera de IQR ({lim_inf:.0f}-{lim_sup:.0f}) imputadas con mediana ({mediana:.0f}): {int(mascara.sum())}")
    return out

### 2.4 Pipeline ETL completo

In [5]:
def aplicar_etl(df):
    pasos = [normalizar_strings, unificar_genero, unificar_estado_entrega,
             parsear_nota, parsear_fechas, remover_duplicados,
             tratar_outliers_nota, tratar_outliers_edad]
    out = df
    for paso in pasos:
        out = paso(out)
    return out

df_limpio = aplicar_etl(df_raw)
print(f"\nFilas: {len(df_limpio)}")

### 2.5 Feature Engineering

#### Decisión de diseño: sensibilidad del Índice de Constancia

El `Indice_Constancia` mide el **porcentaje de actividades previas entregadas *a tiempo*** sobre el total (TPs y Quizzes).

| Situación | Cuenta para `Hizo_Entrega` | Cuenta para `Indice_Constancia` |
|-----------|:--------------------------:|:-------------------------------:|
| Entrega a tiempo | Sí | Sí |
| Entrega tarde | Sí | **No** |
| No entregó | No | No |

**Justificación:** una entrega tardía es mejor que no entregar, pero no equivale a constancia puntual. Esto permite distinguir dos perfiles de riesgo:
- **Alto riesgo:** no entrega (`Entregas_Previas` bajas, `Indice_Constancia` bajo).
- **Riesgo moderado:** entrega sistemáticamente tarde (`Entregas_Previas` altas, `Indice_Constancia` bajo).

La variable `Entregas_Previas` captura el segundo perfil sin necesidad de modificar el índice.

In [ ]:
def feature_engineering(df):
    out = df.copy()

    # Días de anticipación: positivo = entregó antes del límite, negativo = tarde
    out["Dias_Anticipacion"] = (out["Fecha_Limite"] - out["Fecha_Entrega"]).dt.days

    # Validación: fecha de entrega posterior al fin del cuatrimestre es incoherente
    FECHA_FIN_CUATRIMESTRE = pd.Timestamp("2026-06-30")
    entrega_incoherente = out["Fecha_Entrega"] > FECHA_FIN_CUATRIMESTRE
    n_incoherentes = int(entrega_incoherente.sum())
    if n_incoherentes > 0:
        print(
            f"[WARN] {n_incoherentes} entregas con Fecha_Entrega posterior al "
            f"fin del cuatrimestre (2026-06-30) → Dias_Anticipacion marcado como NaN."
        )
        out.loc[entrega_incoherente, "Dias_Anticipacion"] = float("nan")
    else:
        print("[OK] Sin fechas de entrega incoherentes (todas dentro del cuatrimestre).")

    out["Hizo_Entrega"]     = out["Nota"].notna().astype(int)
    out["Entrega_Aprobada"] = (out["Nota"] >= 6).astype("Int64")
    return out

df_limpio = feature_engineering(df_limpio)
df_limpio.head()

### 2.6 Agregación por alumno y exportación

In [ ]:
def agregar_por_alumno(df):
    previas = df[df["Tipo_Actividad"].isin(["Tp","Quiz"])]
    constancia = previas.groupby("ID_Alumno").agg(
        Entregas_Previas=("Hizo_Entrega","sum"),
        Entregas_A_Tiempo=("Estado_Entrega", lambda s: (s=="A Tiempo").sum()),
        Total_Actividades_Previas=("Hizo_Entrega","size"),
        Promedio_Previo=("Nota","mean")
    )
    constancia["Indice_Constancia"] = (100.0 * constancia["Entregas_A_Tiempo"] / constancia["Total_Actividades_Previas"]).round(2)
    parcial = df[df["Tipo_Actividad"]=="Parcial"].groupby("ID_Alumno")["Nota"].first().rename("Nota_Parcial")
    fijos = df.groupby("ID_Alumno").agg(
        Nombre_Apellido=("Nombre_Apellido","first"),
        Edad=("Edad","first"), Genero=("Genero","first"),
        Comision=("Comision","first"), Turno=("Turno","first")
    )
    alumno = fijos.join(constancia).join(parcial).reset_index()
    def _estado(fila):
        n = fila["Nota_Parcial"]
        if pd.isna(n): return "Ausente"
        if n >= 6: return "Aprobado"
        if n >= 4: return "Recupera"
        return "Desaprobado"
    alumno["Estado_Final"] = alumno.apply(_estado, axis=1)
    # Nota: los alias con tildes/espacios se aplican en la capa de visualizacion, no en el CSV.
    return alumno

df_alumno = agregar_por_alumno(df_limpio)
df_alumno.head()

In [8]:
try:
    df_alumno.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
    print(f"[OK] Exportado: {RUTA_SALIDA} ({len(df_alumno)} filas)")
except OSError as e:
    print(f"[ERROR] No se pudo escribir el CSV: {e}")

### 2.7 Verificación final del ETL

In [9]:
print(f"Alumnos únicos:        {df_alumno['ID_Alumno'].nunique()}")
print(f"Comisiones distintas:  {sorted(df_alumno['Comision'].unique())}")
print(f"Turnos distintos:      {sorted(df_alumno['Turno'].unique())}")
print(f"Estados finales:       {df_alumno['Estado_Final'].value_counts().to_dict()}")
print(f"Tasa entrega parcial:  {100*df_alumno['Nota_Parcial'].notna().mean():.1f}%")
print(f"Promedio Nota Parcial: {df_alumno['Nota_Parcial'].mean():.2f}")
print(f"Mediana Indice_Const.: {df_alumno['Indice_Constancia'].median():.1f}%")

---
## Hito 3: Análisis y Visualización

Respondemos las 3 preguntas del Hito 1 con gráficos sobre `df_alumno` (700 alumnos) y `df_limpio` (5.600 entregas).

### Pregunta 1 — ¿El Índice de Constancia predice el abandono del parcial?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Pregunta 1 — Índice de Constancia y rendimiento previo según Estado Final',
             fontsize=13, fontweight='bold')

orden_estados = ['Ausente', 'Desaprobado', 'Recupera', 'Aprobado']
paleta = {'Ausente': '#b0b0b0', 'Desaprobado': '#e74c3c', 'Recupera': '#f39c12', 'Aprobado': '#27ae60'}

# Boxplot: Índice de Constancia por Estado Final
sns.boxplot(data=df_alumno, x='Estado_Final', y='Indice_Constancia',
            order=orden_estados, palette=paleta, ax=axes[0])
axes[0].set_title('Distribución del Índice de Constancia por Estado Final', fontsize=11)
axes[0].set_xlabel('Estado Final')
axes[0].set_ylabel('Índice de Constancia (%)')
mediana_global = df_alumno['Indice_Constancia'].median()
axes[0].axhline(mediana_global, color='navy', linestyle='--', linewidth=1.2,
                label=f'Mediana global: {mediana_global:.1f}%')
axes[0].legend(fontsize=9)

# Scatter: Constancia vs Promedio Previo
for estado in orden_estados:
    sub = df_alumno[df_alumno['Estado_Final'] == estado]
    axes[1].scatter(sub['Indice_Constancia'], sub['Promedio_Previo'],
                    label=estado, color=paleta[estado], alpha=0.55, s=30)
axes[1].set_title('Índice de Constancia vs Promedio Previo (TPs + Quizzes)', fontsize=11)
axes[1].set_xlabel('Índice de Constancia (%)')
axes[1].set_ylabel('Promedio Previo (nota)')
axes[1].axvline(50, color='gray', linestyle=':', linewidth=1)
axes[1].legend(title='Estado Final', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
ausentes  = df_alumno[df_alumno['Estado_Final'] == 'Ausente']['Indice_Constancia']
presentes = df_alumno[df_alumno['Estado_Final'] != 'Ausente']['Indice_Constancia']

print('=== Conclusión — Pregunta 1: Constancia y abandono del parcial ===')
print(f'  Mediana Índice de Constancia → Ausentes: {ausentes.median():.1f}%  |  Presentes: {presentes.median():.1f}%')
print()
print('  Los alumnos que NO se presentaron al parcial tienen un índice de constancia')
print('  significativamente menor al de quienes sí rindieron.')
print('  El boxplot muestra que la mediana de los Ausentes cae ~20 puntos debajo')
print('  de la de los Aprobados. El scatter revela una zona de alto riesgo')
print('  concentrada por debajo del 50% de constancia.')
print('  → Un Índice de Constancia < 43% opera como señal de alerta temprana de abandono.')

### Pregunta 2 — ¿Hay comisiones o turnos significativamente más débiles?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Pregunta 2 — Nota promedio del parcial y tasa de aprobación por Comisión',
             fontsize=13, fontweight='bold')

comisiones_orden = ['A1','A2','A3','B1','B2','C1','C2','C3','C4']
turno_colores    = {'Mañana': '#3498db', 'Tarde': '#e67e22', 'Noche': '#8e44ad'}

# Nota promedio del parcial por Comisión
sns.barplot(data=df_alumno.dropna(subset=['Nota_Parcial']),
            x='Comision', y='Nota_Parcial', hue='Turno',
            order=comisiones_orden, palette=turno_colores,
            errorbar='ci', legend=False, ax=axes[0])
axes[0].set_title('Nota Promedio del Parcial por Comisión', fontsize=11)
axes[0].set_xlabel('Comisión')
axes[0].set_ylabel('Nota promedio (sobre 10)')
axes[0].axhline(6, color='red', linestyle='--', linewidth=1.2, label='Umbral aprobación (6)')
axes[0].legend(fontsize=9)

# Tasa de aprobación por Comisión
tasa = (df_alumno
        .assign(Aprobo=(df_alumno['Estado_Final'] == 'Aprobado').astype(int))
        .groupby(['Comision','Turno'])['Aprobo'].mean()
        .mul(100).reset_index())
sns.barplot(data=tasa, x='Comision', y='Aprobo', hue='Turno',
            order=comisiones_orden, palette=turno_colores, ax=axes[1])
axes[1].set_title('Tasa de Aprobación (%) por Comisión', fontsize=11)
axes[1].set_xlabel('Comisión')
axes[1].set_ylabel('% Alumnos Aprobados')
axes[1].legend(title='Turno', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
tasa_turno = (df_alumno
              .assign(Aprobo=(df_alumno['Estado_Final'] == 'Aprobado').astype(int))
              .groupby('Turno')['Aprobo'].mean().mul(100).round(1))
nota_turno = df_alumno.groupby('Turno')['Nota_Parcial'].mean().round(2)

print('=== Conclusión — Pregunta 2: Diferencias entre comisiones y turnos ===')
print()
print('  Tasa de aprobación y nota promedio por turno:')
for t in ['Mañana', 'Tarde', 'Noche']:
    print(f'    {t:8s}: {tasa_turno[t]:.1f}% aprobados  |  nota prom. parcial: {nota_turno[t]:.2f}')
print()
print('  Las diferencias entre comisiones existen pero son moderadas.')
print('  Ninguna comisión queda sistemáticamente por debajo del umbral de aprobación,')
print('  lo que indica que el problema de abandono es transversal a todos los turnos.')
print('  → La intervención debe ser general, no focalizada en una comisión específica.')

### Pregunta 3 — ¿Cómo evoluciona el rendimiento a lo largo del cuatrimestre?

In [ ]:
# Ordenar actividades cronológicamente por fecha límite mediana
orden_actividades = (
    df_limpio.groupby('Actividad')['Fecha_Limite']
    .median().sort_values().index.tolist()
)

nota_prom = df_limpio.groupby('Actividad')['Nota'].mean().reindex(orden_actividades)
tasa_entr = df_limpio.groupby('Actividad')['Hizo_Entrega'].mean().mul(100).reindex(orden_actividades)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(orden_actividades, nota_prom.values,
         color='steelblue', marker='o', linewidth=2.2, markersize=8, label='Nota promedio')
ax2.plot(orden_actividades, tasa_entr.values,
         color='tomato', marker='s', linewidth=2.2, linestyle='--', markersize=8, label='Tasa de entrega (%)')

ax1.set_xlabel('Actividad (orden cronológico)', fontsize=11)
ax1.set_ylabel('Nota promedio (sobre 10)', color='steelblue', fontsize=11)
ax2.set_ylabel('Tasa de entrega (%)', color='tomato', fontsize=11)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='tomato')
plt.xticks(rotation=30, ha='right')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left', fontsize=10)

plt.title(
    'Pregunta 3 — Evolución de Nota Promedio y Tasa de Entrega por Actividad\n'
    'Grupo 7: Bontorno, Mas, Vernier, Mejias, Hassan, Nuñez',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Conclusión — Pregunta 3: Evolución temporal del rendimiento ===')
print()
print('  El gráfico de doble eje revela dos tendencias clave:')
print()
print('  1. La tasa de entrega cae progresivamente desde los TPs iniciales')
print('     hasta el Parcial, donde solo el ~60% de los alumnos se presentó.')
print()
print('  2. La nota promedio de quienes sí entregan se mantiene relativamente')
print('     estable, lo que indica que el problema no es rendimiento sino abandono.')
print()
print('  → El punto de inflexión se concentra en la mitad del cuatrimestre.')
print('    Una intervención temprana (semanas 4-6) podría reducir los ausentes.')

### Gráfico adicional — Mapa de Correlación entre variables académicas clave

In [ ]:
cols_corr = ['Indice_Constancia', 'Promedio_Previo', 'Nota_Parcial', 'Entregas_Previas']
corr_matrix = df_alumno[cols_corr].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    vmin=-1, vmax=1,
    cbar_kws={'label': 'Coeficiente de correlación de Pearson'}
)
plt.title(
    'Mapa de Correlación — Variables Académicas Clave\n'
    'Grupo 7: Bontorno, Mas, Vernier, Mejias, Hassan, Nuñez',
    fontsize=13, fontweight='bold', pad=15
)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
r_const_nota   = corr_matrix.loc['Indice_Constancia', 'Nota_Parcial']
r_previo_nota  = corr_matrix.loc['Promedio_Previo',   'Nota_Parcial']
r_const_previo = corr_matrix.loc['Indice_Constancia', 'Promedio_Previo']

print('=== Conclusión — Mapa de Correlación ===')
print(f'  Indice_Constancia  ↔ Nota_Parcial   : r = {r_const_nota:.2f}')
print(f'  Promedio_Previo    ↔ Nota_Parcial   : r = {r_previo_nota:.2f}')
print(f'  Indice_Constancia  ↔ Promedio_Previo: r = {r_const_previo:.2f}')
print()
print('  El heatmap confirma cuantitativamente las conclusiones anteriores:')
print('  → La constancia de entregas tiene correlación positiva con la nota del parcial.')
print('  → El promedio previo también correlaciona con el resultado final.')
print('  → Ambas variables juntas construyen el perfil de riesgo del alumno:')
print('     baja constancia + bajo promedio previo = mayor probabilidad de ausencia.')

## Hito 4: Dashboard Interactivo

El dashboard se construyó en **Grafana** (Opción B de la consigna: aplicación web),
conectado a una base **SQLite** (`data/academico.db`) generada a partir del dataset
limpio del Hito 2. Permite filtrar por **Turno** y **Comisión** y actualiza los
KPIs y gráficos en tiempo real.

**Paneles (8):** Tasa de Presentación al Parcial · Alumnos Aprobados · Alumnos en
Riesgo (constancia < 43%) · Total Alumnos filtrados · Distribución de Estados
Finales · Nota Promedio por Comisión · Índice de Constancia (Ausentes vs Presentes)
· Evolución de Nota y Tasa de Entrega por Actividad.

Los archivos del dashboard, los scripts de construcción de la base y las capturas
de pantalla del tablero funcionando están en la carpeta del Hito 4
(`construir_base.py`, `dashboard/dashboard-tpi-grupo7.json`, `notebooks/Capturas/`).
Las instrucciones completas de reproducción están en `HITO4_README.md`.

![Dashboard de Grafana funcionando](Capturas/Dashboard.png)

## Hito 5: Informe de Gestión

*Pendiente.*